Milestone2

In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")

print("Dataset loaded successfully")
df.head()


Dataset loaded successfully


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,NaN,NaN
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,NaN,NaN
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,NaN,NaN
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,NaN,NaN
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,NaN,NaN


In [2]:
df.columns

Index(['id', 'sender', 'subject', 'body', 'priority', 'triage_label',
       'ideal_intent', 'ideal_tone'],
      dtype='object')

In [3]:
df.shape

(200, 8)

In [ ]:

#Create evaluator function 
def evaluate(agent_output, ideal_action, ideal_tone):
    action_match = agent_output.get("action") == ideal_action
    tone_match = agent_output.get("tone") == ideal_tone
    return int(action_match and tone_match)

In [5]:
#Email Assistant Logic
def email_assistant(email_text):
    text = email_text.lower()

    if "urgent" in text or "deadline" in text or "eod" in text:
        return "notify", "urgent"
    elif "thank you" in text or "thanks" in text:
        return "ignore", "polite"
    else:
        return "respond", "neutral"

In [5]:
#Generate Predictions
predictions = []

for _, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    predictions.append({
        "id": row["id"],
        "predicted_intent": action,
        "predicted_tone": tone
    })

pred_df = pd.DataFrame(predictions)
pred_df.head()


,id,predicted_intent,predicted_tone
0,1,respond,neutral
1,2,respond,neutral
2,3,respond,neutral
3,4,respond,neutral
4,5,respond,neutral


In [10]:
df.loc[0, ["ideal_intent", "ideal_tone"]] = ["notify", "neutral"]
df.loc[1, ["ideal_intent", "ideal_tone"]] = ["notify", "urgent"]
df.loc[2, ["ideal_intent", "ideal_tone"]] = ["ignore", "polite"]


In [11]:
#Evaluate Accuracy
eval_df = df.merge(pred_df, on="id")
eval_df.head()


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone,predicted_intent,predicted_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,neutral,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,notify,urgent,respond,neutral
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,ignore,polite,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,,,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,,,respond,neutral


In [ ]:
#Evaluation function 
def evaluate(row):
    score = 0
    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1
    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1
    return score

In [13]:
#Apply evaluation
eval_df["score"] = eval_df.apply(evaluate, axis=1)
eval_df[["body", "ideal_intent", "predicted_intent", "ideal_tone", "predicted_tone", "score"]].head()


,body,ideal_intent,predicted_intent,ideal_tone,predicted_tone,score
0,Reminder: The client meeting is scheduled at 1...,notify,respond,neutral,neutral,1
1,Your invoice of INR 25515.09 is due on 2025-12...,notify,respond,urgent,neutral,0
2,Reminder: The client meeting is scheduled at 1...,ignore,respond,polite,neutral,0
3,"Hello team, please find the attached weekly re...",,respond,,neutral,0
4,"Hello team, please find the attached weekly re...",,respond,,neutral,0


In [ ]:
#Compute accuracy 
accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
accuracy


np.float64(0.25)

In [ ]:
#saving the output
eval_df.to_csv(
    "../data/milestone2_output_theertha.csv",
    index=False
)
